Please note, this is an example script to analyse one membrane replicate simulation, if your membrane system has multiple replicates, you need to adapt this code to input all the replicates and then get 
the average value.

This script was used to compute the ensemble of membrane diffusion coefficient values. It can calculate the diffusion coefficient of the entire membrane, each lealfet (excluding data of the lipids that have flip-flopped), and for every lipid species, but you need to adapt the code to your membrane system. This script will produce .pkl which can be used for later analysis (which is also done in this script).

For analysis, diffusion coefficient of every lipid species is output to .xlsx files, while other results are also printed to the screen

This script leverages the Lipyphilic Python package. For a detailed description of the Lipyphilic package, please see  https://pubs.acs.org/doi/10.1021/acs.jctc.1c00447

In [ ]:
import MDAnalysis as mda
import numpy as np
import sys
import seaborn as sns
import matplotlib.pyplot as plt
import xdrlib
import pickle
import pandas as pd
import tqdm

from lipyphilic.transformations import nojump
from lipyphilic.lib.lateral_diffusion import MSD
%matplotlib inline

In [ ]:
u = mda.Universe(
    "PATH/TO/YOUR/TPR/FILE.tpr",  # topology file 
    "PATH/TO/YOUR/XTC/FILE.xtc"   # trajectory file
)

In [ ]:
ag = u.select_atoms("name GL1 GL2 AM1 AM2  ROH ES")
membrane = u.select_atoms("name PO4 GL1 GL2 AM1 AM2 ES ROH").residues

In [ ]:
u.trajectory.add_transformations(
    nojump(
        ag=ag,
        nojump_x=True,
        nojump_y=True,
        nojump_z=False)
)

In [ ]:
msd = MSD(
    universe=u,
    lipid_sel="name GL1 GL2 AM1 AM2  ES ROH",
    com_removal_sel="name GL1 GL2 AM1 AM2  ES ROH"
)

In [ ]:
msd.run(
    start=None,
    stop=None,
    step=None,
    verbose=True
)

In [ ]:
with open ("MSD.pkl", "wb") as f:
          pickle.dump(msd, f)

In [ ]:
with open ("MSD.pkl", "rb") as f:
    original_MSD=pickle.load(f)

print(all(original_MSD.msd.T[0] == 0))

In [ ]:
## If you have muliple replicates, you and average the their original MSD point value here, using np.mean(np.array([array1,...]), axis=0)
## Then you need to create a new MSD based on the averaged value
## Please change code accordingly
# lipid_sel = "name GL1 GL2 AM1 AM2 ROH ES"  
# com_removal_sel = "name GL1 GL2 AM1 AM2 ROH ES"  # Example, replace with your actual selection
# wholeUniverse = original_MSD.u # Use the same universe from the original instance
# #dt = msd_upper_r1.dt  # Use the same time step   500ps
# 
# msd_whole_avg = MSD(original_MSD.u, lipid_sel, com_removal_sel=com_removal_sel, dt=original_MSD.dt)
# 
# 
# msd_whole_avg.msd = whole_average 
# msd_whole_avg.lagtimes = original_MSD.lagtimes  


## Retrieve the residue indices from the original MSD instances
#indices_r1 = original_MSD_r1.membrane.residues.resindices
#indices_r2 = original_MSD_r2.membrane.residues.resindices
#indices_r3 = original_MSD_r3.membrane.residues.resindices
#
## Retrieve the residue indices from the averaged MSD instance
#indices_avg = msd_whole_avg.membrane.residues.resindices
#
## Check if the ordering is consistent
#ordering_consistent_r1 = np.array_equal(indices_avg, indices_r1)
#ordering_consistent_r2 = np.array_equal(indices_avg, indices_r2)
#ordering_consistent_r3 = np.array_equal(indices_avg, indices_r3)
#
#print(f"Ordering consistent with replicate 1: {ordering_consistent_r1}")
#print(f"Ordering consistent with replicate 2: {ordering_consistent_r2}")
#print(f"Ordering consistent with replicate 3: {ordering_consistent_r3}")

In [ ]:
plt.figure(figsize=(10, 10))
for species in np.unique(membrane.resnames):

    # create a boolean mask for filterin out the current species
    species_mask = (membrane.residues.resnames == species)
    
    # select the msd of the current species
    species_msd = original_MSD.msd[species_mask]  # this contains the msd of each lipid molecule of the relevant species
    #print(species_msd.shape)
    # get the mean MSD for the species
    mean_species_msd = np.mean(species_msd, axis=0)
    
    # Plot the MSD against lagtime
    plt.loglog(
        original_MSD.lagtimes,
        mean_species_msd,
        label=species
    )
 

plt.xlabel("Lagtime (ns)", fontsize=14)
plt.ylabel(r"MSD$_{xy}\ \rm{(nm^2)}$", fontsize=14)

#plt.legend()


In [ ]:
d, sem  = original_MSD.diffusion_coefficient(
    start_fit=100,  # start fitting from a lagtime of 100 ns    
    stop_fit=1000   # stop fitting from a lagtime of 1000ns
)

print("whole membrane")
print(f" mean: {d:.3}, sem: {sem:.3}")


In [ ]:
d_noChol_Chyo, sem_noChol_Chyo = original_MSD.diffusion_coefficient(
    start_fit=100,  # start fitting from a lagtime of 100 ns    
    stop_fit=1000,   # stop fitting from a lagtime of 1000ns
    lipid_sel="name GL1 GL2 AM1 AM2"  # exclude choletserol and cholesterol ester
)
print(f" mean: {d_noChol_Chyo:.3}, sem: {d_noChol_Chyo:.3}")


In [ ]:
# leaflet value
print("upper")
d_upper, sem_upper  = original_MSD.diffusion_coefficient(
    start_fit=100,  # start fitting from a lagtime of 100 ns    
    stop_fit=1000,   # stop fitting from a lagtime of 1000ns
    lipid_sel="name GL1 GL2 AM1 AM2 and resid 1 to 1900"    #Here, we need to select lipids (exclude chol and chyo) that are in upper leaflet, the resid need to change accordingly to your gro file, same for the rest.
)                                                           
print("r1 upper" + f" mean: {d_upper:.3}, sem: {sem_upper:.3}")

In [ ]:
# leaflet value
print("lower")
d_lower, sem_lower  = original_MSD.diffusion_coefficient(
    start_fit=100,  # start fitting from a lagtime of 100 ns    
    stop_fit=1000,   # stop fitting from a lagtime of 1000ns
    lipid_sel="name GL1 GL2 AM1 AM2 and resid 1900 to 4500"
)
print("r1 lower" + f" mean: {d_lower:.3}, sem: {sem_lower:.3}")

In [ ]:
print("whole membrane")
for species in ["*PC", "*PE", "*SM", "*PI", "*PS", "CHOL", "CHYO"]: #np.unique(membrane.resnames):
    
    d_hg, sem_hg = original_MSD.diffusion_coefficient(
        start_fit=100,
        stop_fit=1000,
        lipid_sel=f"resname {species}"
    )

    print(f"species: {species}, mean: {d_hg:.3}, sem: {sem_hg:.3}")


In [ ]:
# ignore the warning, because upper leaflet doesn't have PI and PS
print(" upper leaflet ")
for species in ["*PC", "*PE", "*SM", "*PI", "*PS"]: #np.unique(membrane.resnames):
    
    d_hg_u, sem_hg_u = original_MSD.diffusion_coefficient(
        start_fit=100,
        stop_fit=1000,
        lipid_sel=f"resname {species} and resid 1 to 1900"
    )

    print(f"species: {species}, mean: {d_hg_u:.3}, sem: {sem_hg_u:.3}")


In [ ]:
print(" lower leaflet ")
for species in ["*PC", "*PE", "*SM", "*PI", "*PS"]: #np.unique(membrane.resnames):
    
    d_hg_l, sem_hg_l = original_MSD.diffusion_coefficient(
        start_fit=100,
        stop_fit=1000,
        lipid_sel=f"resname {species} and resid 1900 to 4500"
    )

    print(f"species: {species}, mean: {d_hg_l:.3}, sem: {sem_hg_l:.3}")



Calculate MSD based on unsaturation degree on each leaflet 

In [ ]:
# Please change the lipid name and resid according to your membrane
print(" upper unsaturation ") 
d_s_avg, sem_s_avg = original_MSD.diffusion_coefficient(
    start_fit=100,
    stop_fit=1000,
    lipid_sel="resname LPPC DPPC DPPS and resid 1 to 1900"
)

print("avg Sat mean:" + str(d_s_avg) + ", sem:" + str(sem_s_avg))

d_m_avg, sem_m_avg = original_MSD.diffusion_coefficient(
    start_fit=100,
    stop_fit=1000,
    lipid_sel="resname LVPC LOPC VLPC OLPC PVPC POPC VPPC OPPC PGPC LOPE POPE PVPE PGPE POPS LGPS PVPS PGPS POPI PVPI DPSM PBSM PXSM and resid 1 to 1900"
)

print("avg Mono mean:" + str(d_m_avg) + ", sem:" + str(sem_m_avg))

d_p_avg, sem_p_avg = original_MSD.diffusion_coefficient(
    start_fit=100,
    stop_fit=1000,
    lipid_sel="resname DVPC DOPC PZPC PIPC PSPC PEPC PCPC PQPC PIPE DOPE PSPE PEPE PCPE PQPE PAPE PUPE POSM PGSM PNSM PMSM PIPS PSPS PEPS PCPS PQPS PUPS PIPI PSPI PEPI PCPI PQPI PAPI PUPI and resid 1 to 1900"
)

print("avg Poly mean:" + str(d_p_avg) + ", sem:" + str(sem_p_avg))

In [ ]:
print(" lower unsaturation ") 
d_s_avg, sem_s_avg = original_MSD.diffusion_coefficient(
    start_fit=100,
    stop_fit=1000,
    lipid_sel="resname LPPC DPPC DPPS and resid 1900 to 4500"
)

print("avg Sat mean:" + str(d_s_avg) + ", sem:" + str(sem_s_avg))

d_m_avg, sem_m_avg = original_MSD.diffusion_coefficient(
    start_fit=100,
    stop_fit=1000,
    lipid_sel="resname LVPC LOPC VLPC OLPC PVPC POPC VPPC OPPC PGPC LOPE POPE PVPE PGPE POPS LGPS PVPS PGPS POPI PVPI DPSM PBSM PXSM and resid 1900 to 4500"
)

print("avg Mono mean:" + str(d_m_avg) + ", sem:" + str(sem_m_avg))

d_p_avg, sem_p_avg = original_MSD.diffusion_coefficient(
    start_fit=100,
    stop_fit=1000,
    lipid_sel="resname DVPC DOPC PZPC PIPC PSPC PEPC PCPC PQPC PIPE DOPE PSPE PEPE PCPE PQPE PAPE PUPE POSM PGSM PNSM PMSM PIPS PSPS PEPS PCPS PQPS PUPS PIPI PSPI PEPI PCPI PQPI PAPI PUPI and resid 1900 to 4500"
)

print("avg Poly mean:" + str(d_p_avg) + ", sem:" + str(sem_p_avg))

calculate every single lipid species in upper and lower leaflet

In [ ]:
print(" upper leaflet lipid species MSD ")
data_UL = []
for species in np.unique(membrane.resnames):
    
    d_hg_each_u, sem_hg_each_u = original_MSD.diffusion_coefficient(
        start_fit=100,
        stop_fit=1000,
        lipid_sel=f"resname {species} and resid 1 to 1900"
    )

    print(f"species: {species}, mean: {d_hg_each_u:.3}, sem: {sem_hg_each_u:.3}")
    mean_scaled = d_hg_each_u * 1e7  # Scale by 1e7
    sem_scaled = sem_hg_each_u * 1e7  # Scale by 1e7
    data_UL.append([species, f"{mean_scaled:.2f} ± {sem_scaled:.2f}"])
f = pd.DataFrame(data_UL, columns=["Species", "UL MSD(Mean ± SEM)"])
f.to_excel("Membrane_UL_MSD.xlsx", index=False)   

In [ ]:
print(" lower leaflet lipid species MSD ")
data_LF = []
for species in np.unique(membrane.resnames):
    
    d_hg_each_l, sem_hg_each_l = original_MSD.diffusion_coefficient(
        start_fit=100,
        stop_fit=1000,
        lipid_sel=f"resname {species}  and resid 1900 to 4500"
    )
    print(f"species: {species}, mean: {d_hg_each_l:.3}, sem: {sem_hg_each_l:.3}")
    mean_scaled = d_hg_each_l * 1e7  # Scale by 1e7
    sem_scaled = sem_hg_each_l * 1e7  # Scale by 1e7
    data_LF.append([species, f"{mean_scaled:.2f} ± {sem_scaled:.2f}"])
df = pd.DataFrame(data_LF, columns=["Species", "LL MSD(Mean ± SEM)"])
df.to_excel("Membrane_LL_MSD.xlsx", index=False)   
    
